In [1]:
%load_ext autoreload
%autoreload 3 --print --log

# 位姿、外参与坐标约定

本例不需要数据集。使用 AIO 内核，或先安装 graphmap 与本地 scipykit。
`T_world_camera` 把相机点送到世界；乘法右侧先执行。
局部 X 前、Y 左、Z 上，四元数 xyzw，欧拉角默认度。

In [2]:
import numpy as np
from scipykit.task.cv import *
from graphmap.pose import Pose, PoseArray
from graphmap.pose.conventions import convert_points, convert_pose

T_world_body = Pose(x=1, y=2, yaw=35, wrd_frame="world", ego_frame="body")
T_body_camera = Pose(x=0.3, z=1, pitch=-10, wrd_frame="body", ego_frame="camera")
T_world_camera = T_world_body * T_body_camera
points_camera = np.array([[2., 0., 0.], [2., 0.5, 0.], [2., 0., 0.5]])
points_world = T_world_camera(points_camera)
np.testing.assert_allclose((~T_world_camera)(points_world), points_camera, atol=1e-12)
print(T_world_camera)
print(points_world)

Pose(pos=[1.246 2.172 1.   ], euler_deg=[  0. -10.  35.], scale=1) [world <- camera]
[[2.85916018 3.30179797 1.34729636]
 [2.57237196 3.711374   1.34729636]
 [2.78803805 3.25199772 1.83970023]]


## 画出相机坐标轴
红/绿/蓝分别为 X/Y/Z。这里只画真实三维坐标，不使用图像 h/w 的轴顺序。

In [7]:
fig = plt.figure(figsize=(7, 5), dpi=120)
ax = fig.add_subplot(111, projection="3d")
origin = T_world_camera.position
for axis, color, label in zip(np.eye(3), ["r", "g", "b"], ["X forward", "Y left", "Z up"]):
    end = T_world_camera(axis * 0.7)
    ax.plot(*np.stack([origin, end]).T, color=color, label=label)
ax.scatter(*points_world.T, color="black", label="world points")
ax.set(xlabel="world x (m)", ylabel="world y (m)", zlabel="world z (m)")
ax.legend()
disp(fig, 'tmp')
plt.close(fig)

/gray/Storage/Repositories/graphmap/examples/assets/01_pose_and_frames.ipynb/tmp.png

## 批量位姿与三种应用方式
`apply_each` 是逐点广播，`apply_clouds` 为每个位姿对应一份点云，`apply_all` 将同一份点云应用到全部位姿。

In [4]:
trajectory = PoseArray(position=[[0,0,0], [1,0,0], [2,0,0]], rotation=[[0,0,0], [0,0,30], [0,0,60]])
clouds = trajectory.apply_all(points_camera)
assert clouds.shape == (3, 3, 3)
np.testing.assert_allclose(
    trajectory.apply_clouds(clouds, inverse=True),
    np.broadcast_to(points_camera, clouds.shape), atol=1e-12,
)
print("pose shape:", trajectory.shape, "cloud shape:", clouds.shape)

pose shape: (3,) cloud shape: (3, 3, 3)


## 外部坐标基与外参方向
OpenCV 点 `[right, down, forward]` 转成 graphmap `[forward, left, up]`。
位姿需要分别转换世界与相机两端的基：`C_world @ T @ inv(C_local)`。
以下世界系原本已经是 graphmap，只有局部相机使用 OpenCV。

In [5]:
p_cv = np.array([[0.2, 0.3, 2.]])
p_graph = convert_points(p_cv, source="opencv")
np.testing.assert_allclose(p_graph, [[2., -0.2, -0.3]])
T_world_optical = Pose(position=[1,2,3], rotation=[10,20,30])
T_world_graph_camera = convert_pose(T_world_optical, world="graphmap", local="opencv")
np.testing.assert_allclose(T_world_graph_camera(p_graph), T_world_optical(p_cv))
print("OpenCV:", p_cv, "graphmap:", p_graph)

OpenCV: [[0.2 0.3 2. ]] graphmap: [[ 2.  -0.2 -0.3]]


## 相似变换
`Pose(scale=...)` 可用于点云对齐；把输入送入米制建图适配器前，需要先完成尺度转换。传感器安装外参应保持 scale=1。

In [6]:
similarity = Pose.from_about(rotation=[0,0,90], scale=2, about=[1,0,0])
np.testing.assert_allclose(similarity([1,0,0]), [1,0,0])
np.testing.assert_allclose(similarity([2,0,0]), [1,2,0], atol=1e-12)
assert Pose.from_matrix(similarity.matrix).allclose(similarity)
print(similarity)

Pose(pos=[ 1. -2.  0.], euler_deg=[ 0.  0. 90.], scale=2) [None <- None]
